# 🔍 DefectVision — 04. PatchCore (Anomaly Detection)

**Objectif** : Entraîner PatchCore pour détecter les anomalies industrielles sur MVTec AD.

**Workflow** :
```
Train (good only) → WideResNet50 features → Memory Bank → Score anomalie → AUROC
```

**Pourquoi PatchCore après YOLOv8 ?**
> YOLOv8 (supervisé) a obtenu mAP50=0.09 faute de données défectueuses.
> PatchCore (non supervisé) s'entraîne uniquement sur les images normales
> et atteint AUROC ~99% sur MVTec AD — c'est le state-of-the-art.

## 0. Setup

In [ ]:
!nvidia-smi

In [ ]:
# Anomalib — librairie Intel pour la détection d'anomalies
!pip install -q anomalib==1.1.0
!pip install -q huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_BASE  = Path('/content/drive/MyDrive/defect-vision')
MVTEC_PATH  = DRIVE_BASE / 'data' / 'mvtec'
MODELS_PATH = DRIVE_BASE / 'models' / 'patchcore_bottle'
MODELS_PATH.mkdir(parents=True, exist_ok=True)

CATEGORY = 'bottle'

print(f'✅ MVTec path  : {MVTEC_PATH}')
print(f'✅ Models path : {MODELS_PATH}')
assert (MVTEC_PATH / CATEGORY).exists(), '❌ Dataset introuvable — lance le notebook 01 dabord'

## 1. Vérification des données

In [ ]:
import os

cat_path = MVTEC_PATH / CATEGORY

train_good = list((cat_path / 'train' / 'good').glob('*.png'))
test_good  = list((cat_path / 'test'  / 'good').glob('*.png'))
test_defects = {}
for d in (cat_path / 'test').iterdir():
    if d.is_dir() and d.name != 'good':
        test_defects[d.name] = list(d.glob('*.png'))

total_defects = sum(len(v) for v in test_defects.values())

print(f'=== {CATEGORY} ===')
print(f'  Train good     : {len(train_good)} images  ← utilisées pour la memory bank')
print(f'  Test good      : {len(test_good)} images')
print(f'  Test defects   : {total_defects} images ({list(test_defects.keys())})')
print(f'\n✅ PatchCore utilisera uniquement les {len(train_good)} images good pour sentraîner')

## 2. Configuration PatchCore

In [ ]:
from anomalib.models import Patchcore
from anomalib.data import MVTec
from anomalib.engine import Engine
from anomalib.data.utils import TestSplitMode
import torch

print(f'✅ Anomalib importé')
print(f'✅ GPU disponible : {torch.cuda.is_available()}')
print(f'✅ Device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Datamodule MVTec
datamodule = MVTec(
    root          = str(MVTEC_PATH),
    category      = CATEGORY,
    image_size    = (224, 224),
    train_batch_size = 32,
    eval_batch_size  = 32,
    num_workers   = 2,
    test_split_mode = TestSplitMode.FROM_DIR,
)

print('✅ Datamodule configuré')

In [ ]:
# Modèle PatchCore
model = Patchcore(
    backbone        = 'wide_resnet50_2',  # backbone pré-entraîné ImageNet
    layers_list     = ['layer2', 'layer3'],
    num_neighbors   = 9,
    coreset_sampling_ratio = 0.1,         # 10% des patches gardés en mémoire
)

print('✅ Modèle PatchCore configuré')
print('  Backbone    : wide_resnet50_2')
print('  Layers      : layer2 + layer3')
print('  Neighbors   : 9')
print('  Coreset     : 10%')

## 3. Entraînement (construction de la Memory Bank)

In [ ]:
import time

# Engine Anomalib
engine = Engine(
    max_epochs  = 1,   # PatchCore n'a qu'une seule passe (pas de backprop)
    accelerator = 'gpu',
    devices     = 1,
    default_root_dir = '/content/anomalib_runs',
)

print('🚀 Construction de la Memory Bank...')
print('   (PatchCore nextraie que les features — pas de backpropagation)')
start = time.time()

engine.fit(
    model      = model,
    datamodule = datamodule,
)

elapsed = time.time() - start
print(f'\n✅ Memory Bank construite en {elapsed:.1f} secondes')

## 4. Évaluation

In [ ]:
print('📊 Évaluation sur le test set...')

test_results = engine.test(
    model      = model,
    datamodule = datamodule,
)

print('\n=== Métriques PatchCore ===')
for key, val in test_results[0].items():
    print(f'  {key:30s} : {val:.4f}')

## 5. Visualisation des heatmaps d'anomalies

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
import torchvision.transforms as T
import random

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

def predict_anomaly(img_path):
    img = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
    score   = output.pred_score.item()
    heatmap = output.anomaly_map.squeeze().cpu().numpy()
    return img, score, heatmap

print('✅ Fonction danalyse prête')

In [ ]:
# Récupérer des exemples good et defect
good_imgs   = random.sample(list((cat_path / 'test' / 'good').glob('*.png')), 3)
defect_imgs = []
for defect_type, imgs in test_defects.items():
    defect_imgs.append(random.choice(imgs))
    if len(defect_imgs) == 3:
        break

all_samples = [('good', p) for p in good_imgs] + [('defect', p) for p in defect_imgs]

fig, axes = plt.subplots(3, 6, figsize=(24, 12))
fig.suptitle('PatchCore — Heatmaps d anomalies | Good (gauche) vs Defect (droite)',
             fontsize=14, fontweight='bold')

row_titles = ['Image originale', 'Heatmap anomalie', 'Overlay']
for row_idx, row_title in enumerate(row_titles):
    axes[row_idx, 0].set_ylabel(row_title, fontsize=11, fontweight='bold')

for col_idx, (label, img_path) in enumerate(all_samples):
    img, score, heatmap = predict_anomaly(img_path)
    img_np = np.array(img.resize((224, 224)))

    # Normaliser heatmap
    heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    heatmap_colored = cm.jet(heatmap_norm)[:, :, :3]

    # Overlay
    overlay = (0.6 * img_np / 255 + 0.4 * heatmap_colored)
    overlay = np.clip(overlay, 0, 1)

    border_color = '#e74c3c' if label == 'defect' else '#2ecc71'

    # Ligne 0 : image originale
    axes[0, col_idx].imshow(img_np)
    axes[0, col_idx].set_title(f'{label.upper()}\nscore={score:.3f}',
                                fontsize=9, fontweight='bold',
                                color='red' if label == 'defect' else 'green')
    axes[0, col_idx].axis('off')

    # Ligne 1 : heatmap
    axes[1, col_idx].imshow(heatmap_colored)
    axes[1, col_idx].axis('off')

    # Ligne 2 : overlay
    axes[2, col_idx].imshow(overlay)
    axes[2, col_idx].axis('off')

plt.tight_layout()
plt.savefig('patchcore_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Heatmaps sauvegardées : patchcore_heatmaps.png')

## 6. Comparaison YOLOv8 vs PatchCore

In [ ]:
yolo_map50    = 0.0933
yolo_recall   = 0.8095
yolo_precision = 0.0028

# Récupérer les métriques PatchCore
pc_auroc = test_results[0].get('image_AUROC', test_results[0].get('AUROC', 0))
pc_f1    = test_results[0].get('image_F1Score', test_results[0].get('F1Score', 0))

print('=' * 55)
print(f'{"Métrique":25s} {"YOLOv8n":15s} {"PatchCore":15s}')
print('=' * 55)
print(f'{"Approche":25s} {"Supervisé":15s} {"Non supervisé":15s}')
print(f'{"Données train":25s} {"209 imgs (6 def.)":15s} {"209 imgs (good)": 15s}')
print(f'{"mAP50":25s} {yolo_map50:.4f}{"":10s} {"N/A":15s}')
print(f'{"Precision":25s} {yolo_precision:.4f}{"":10s} {"N/A":15s}')
print(f'{"Recall":25s} {yolo_recall:.4f}{"":10s} {"N/A":15s}')
print(f'{"AUROC":25s} {"N/A":15s} {pc_auroc:.4f}')
print(f'{"F1 Score":25s} {"N/A":15s} {pc_f1:.4f}')
print('=' * 55)
print()
print('Conclusion : PatchCore est bien adapté à MVTec AD')
print('car il nexige pas de données défectueuses annotées.')

## 7. Sauvegarde sur Drive

In [ ]:
import shutil

# Trouver le checkpoint Anomalib
run_dir = Path('/content/anomalib_runs')
ckpts   = list(run_dir.rglob('*.pt')) + list(run_dir.rglob('*.ckpt'))

print(f'Checkpoints trouvés : {len(ckpts)}')
for ck in ckpts:
    dest = MODELS_PATH / ck.name
    shutil.copy2(ck, dest)
    print(f'  ✅ Sauvegardé : {dest}')

# Sauvegarder la heatmap
shutil.copy2('patchcore_heatmaps.png', MODELS_PATH / 'patchcore_heatmaps.png')
print(f'\n✅ Tout sauvegardé dans : {MODELS_PATH}')

## 8. Push sur HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi, login

# ⚠️ Ne jamais committer ce token sur GitHub !
HF_TOKEN    = 'TON_TOKEN_HUGGINGFACE'  # remplace ici
HF_USERNAME = 'Chasston'
REPO_NAME   = 'defect-vision-patchcore-bottle'

login(token=HF_TOKEN)
api     = HfApi()
repo_id = f'{HF_USERNAME}/{REPO_NAME}'

api.create_repo(repo_id=repo_id, repo_type='model', exist_ok=True)
print(f'✅ Repo : {repo_id}')

In [ ]:
readme = f"""---
tags:
- anomaly-detection
- computer-vision
- patchcore
- defect-detection
- industrial
datasets:
- mvtec-ad
---

# DefectVision — PatchCore Anomaly Detection (bottle)

PatchCore trained on MVTec AD for unsupervised industrial defect detection.

## Why PatchCore over YOLOv8?
YOLOv8 (supervised) achieved mAP50=0.09 due to lack of labeled defect data.
PatchCore (unsupervised) trains only on normal images and achieves state-of-the-art AUROC.

## Category: bottle
- **Defects**: broken_large, broken_small, contamination
- **Backbone**: WideResNet50
- **Training**: good images only ({len(train_good)} images)

## Metrics (test set)
- AUROC   : {pc_auroc:.4f}
- F1 Score: {pc_f1:.4f}

## Author
[{HF_USERNAME}](https://huggingface.co/{HF_USERNAME})
"""

with open('/content/README_patchcore.md', 'w') as f:
    f.write(readme)

# Upload
files = list(MODELS_PATH.glob('*')) + [Path('/content/README_patchcore.md')]
for fp in files:
    fname = 'README.md' if fp.name == 'README_patchcore.md' else fp.name
    api.upload_file(
        path_or_fileobj = str(fp),
        path_in_repo    = fname,
        repo_id         = repo_id,
        repo_type       = 'model',
    )
    print(f'  ✅ {fname}')

print(f'\n🎉 https://huggingface.co/{repo_id}')

## ✅ Résumé

| Info | Valeur |
|------|--------|
| Modèle | PatchCore (WideResNet50) |
| Dataset | MVTec AD — bottle |
| Approche | Non supervisée (good only) |
| Drive | `/defect-vision/models/patchcore_bottle/` |
| HuggingFace | `Chasston/defect-vision-patchcore-bottle` |
| Prochain notebook | `05_fastapi_gradio.ipynb` |

**Prochaine étape** : Créer l'API FastAPI + interface Gradio.